# Bloom Agent — Teste da Ponte (Colab ↔ PC)

Este notebook testa a ponte: lê o job criado no addon, baixa as entradas, gera um resultado e devolve pelo túnel.

**Passos:**
1. No Blender (painel N > Bloom): **Iniciar/parar ponte** → rode no seu PC `ngrok http 8791` → **Job de teste**.
2. Cole aqui embaixo o `job_id` e a `base_url` que aparecem no log do painel.
3. Execute todas as células. Se no final aparecer `PONTE OK`, no Blender clique em **Coletar** com o mesmo job_id.

> O ngrok free tem 1 GB/mês na direção do PC — por isso resultados > 150 MB são salvos no Google Drive (célula de envio) e o addon baixa dali.

In [ ]:
job_id = ""    # ex.: 20260924-153012-a1b2c3
base_url = ""  # ex.: https://xxxx.ngrok-free.app  (ou http://127.0.0.1:8791 p/ teste local)

In [ ]:
import json
import urllib.request

assert job_id and base_url, "Preencha job_id e base_url acima."
base_url = base_url.rstrip("/")

def get(url: str) -> bytes:
    with urllib.request.urlopen(url, timeout=60) as r:
        return r.read()

def post(url: str, data: bytes, headers: dict | None = None) -> dict:
    req = urllib.request.Request(url, data=data, method="POST", headers=headers or {})
    with urllib.request.urlopen(req, timeout=60) as r:
        return json.loads(r.read().decode("utf-8"))

print("health:", json.loads(get(base_url + "/health").decode("utf-8")))
job = json.loads(get(f"{base_url}/job/{job_id}").decode("utf-8"))
print("job:", json.dumps(job, indent=2, ensure_ascii=False))

In [ ]:
# Baixa os arquivos de entrada (ex.: a foto de referência) para /content/bloom_in/
import os

os.makedirs("/content/bloom_in", exist_ok=True)
inputs = []
for name in job.get("input_files", []) or []:
    data = get(f"{base_url}/job/{job_id}/input/{name}")
    path = f"/content/bloom_in/{name}"
    with open(path, "wb") as f:
        f.write(data)
    inputs.append((name, len(data)))
print("entradas baixadas:", inputs or "(nenhuma)")

In [ ]:
# Gera o resultado do kind. 'echo' = teste puro. Outros kinds entram nos próximos notebooks.
from PIL import Image

kind = job.get("kind", "echo")
results = {}  # nome -> bytes

if kind == "echo":
    results["echo.txt"] = f"ok: job {job_id} kind {kind} params {job.get('params')}".encode("utf-8")
    img = Image.new("RGB", (128, 128), (120, 200, 90))
    from PIL import ImageDraw
    d = ImageDraw.Draw(img)
    d.rectangle([16, 16, 112, 112], outline=(20, 20, 20), width=6)
    d.ellipse([48, 48, 80, 80], fill=(250, 240, 120))
    import io
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    results["ping.png"] = buf.getvalue()
else:
    raise NotImplementedError(f"kind {kind!r} ainda não implementado neste notebook (M0).")

print({k: len(v) for k, v in results.items()})

In [ ]:
# Envia os resultados: via ngrok se pequeno; Drive se grande (> 150 MB, por causa da cota free)
DRIVE_LIMIT = 150_000_000
sent = []

for name, data in results.items():
    if len(data) <= DRIVE_LIMIT:
        resp = post(f"{base_url}/job/{job_id}/result", data, {"X-Result-Name": name})
        print(name, "-> ngrok:", resp)
    else:
        from google.colab import drive
        drive.mount("/content/drive", force=True)
        folder = f"/content/drive/MyDrive/{job.get('drive_folder', '3d-create/jobs/' + job_id)}"
        os.makedirs(folder, exist_ok=True)
        with open(os.path.join(folder, name), "wb") as f:
            f.write(data)
        print(name, "-> Drive:", folder)
    sent.append(name)

complete = post(f"{base_url}/job/{job_id}/complete",
                json.dumps({"status": "ok", "files": sent}).encode("utf-8"),
                {"Content-Type": "application/json"})
print("complete:", complete)
print("\n✅ PONTE OK — no Blender, clique em Coletar com job_id", job_id)